In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
!ls

In [ ]:
!unzip glove.6B.zip

In [ ]:
!ls

In [ ]:
!mkdir glove && mv glove*.txt glove

In [ ]:
!ls glove

In [ ]:
!head -n 20 glove/glove.6B.50d.txt

In [ ]:
! pip install emoji

In [ ]:
import emoji

In [ ]:
emoji.EMOJI_UNICODE.keys()

In [ ]:
emoji.EMOJI_UNICODE["en"]

In [ ]:
print(emoji.emojize(":1st_place_medal:")) # returns the unicode character corresponding to string key

In [ ]:
type(emoji)

In [ ]:
!wget https://raw.githubusercontent.com/coding-blocks-archives/machine-learning-online-2018/master/20.%20RNN/Emoji%20Predictor/dataset/train_emoji.csv https://raw.githubusercontent.com/coding-blocks-archives/machine-learning-online-2018/master/20.%20RNN/Emoji%20Predictor/dataset/test_emoji.csv

In [ ]:
!ls

# Getting the dataset

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv("train_emoji.csv", header=None)
train_df.head()

In [ ]:
test_df = pd.read_csv("test_emoji.csv", header=None)
test_df.head()

In [ ]:
# last 2 columns are not useful in train df
train_df = train_df.drop([2, 3], axis=1)
train_df

# also, see that every phrase in test df is having \t character at end, so remove that as well
test_df[0] = test_df[0].apply(lambda text:text[:-1])
test_df

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
training_corpus = train_df[0].values
y_train = train_df[1].values

testing_corpus = test_df[0].values
y_test = test_df[1].values

In [ ]:
training_corpus

In [ ]:
training_corpus, y_train, testing_corpus, y_test

In [ ]:
emoji_dictionary = {
    "0": "\u2764\uFE0F",    # :heart: prints a black instead of red heart depending on the font
    "1": ":baseball:",
    "2": ":beaming_face_with_smiling_eyes:",
    "3": ":downcast_face_with_sweat:",
    "4": ":fork_and_knife:",
    }

In [ ]:
emoji.emojize(emoji_dictionary["0"]), emoji.emojize(emoji_dictionary["4"])

# Data Preparation

In [ ]:
with open("glove/glove.6B.50d.txt", encoding="utf-8") as file:
  glove_list = file.readlines()

In [ ]:
len(glove_list)

In [ ]:
glove_list[0:5]

In [ ]:
wv = {}
for vector in glove_list:
  vector = vector.split()
  wv[vector[0]] = np.array(vector[1:])

In [ ]:
# wv

In [ ]:
wv["eat"]

In [ ]:
def create_dense(corpus, wv, max_len=10):
  X = np.zeros((len(corpus), max_len, 50))

  for i, sentence in enumerate(corpus):

    for tau, word in enumerate(sentence.split()):
      if tau >= max_len:
        break
      
      word = word.lower()
      try:
        X[i][tau] = wv[word]
      except:
        pass

  return X

In [ ]:
X_train = create_dense(training_corpus, wv)

In [ ]:
X_train.shape

In [ ]:
X_test = create_dense(testing_corpus, wv)

In [ ]:
X_test.shape

In [ ]:
X_train

In [ ]:
from tensorflow.keras.utils import to_categorical

In [ ]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# Model Preparation

### LSTM based model

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense

In [ ]:
model = Sequential()

In [ ]:
model.add(LSTM(64, input_shape=(10, 50)))
model.add(Dense(5, activation='softmax'))

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import ModelCheckpoint
model_checkpoint_callback = ModelCheckpoint('best_emoji_weights.h5', monitor='val_accuracy', save_best_only=True)

In [ ]:
# from keras.callbacks import EarlyStopping
# early_stopping_callback = EarlyStopping(monitor='val_accuracy', min_delta=0.01, patience=15, restore_best_weights=True)

In [ ]:
hist = model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[model_checkpoint_callback])

In [ ]:
model.load_weights("best_emoji_weights.h5")

In [ ]:
plt.plot(hist.history["accuracy"])
plt.plot(hist.history["val_accuracy"])
plt.show()

In [ ]:
model.evaluate(X_train, y_train)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
pred = model.predict(X_test)
pred

In [ ]:
y_pred = np.argmax(pred, axis=1)
y_pred

In [ ]:
# print the prediction vs actual emoji 
for i, sent in enumerate(testing_corpus):
  print(sent + emoji.emojize(emoji_dictionary[str(y_pred[i])]) + emoji.emojize(emoji_dictionary[str(np.argmax(y_test, axis=1)[i])]))
  print()

In [ ]:
model_checkpoint_callback

In [ ]:
for i, sent in enumerate(testing_corpus):
  predicted_emoji_code = emoji_dictionary[str(y_pred[i])]
  predicted_emoji = emoji.emojize(predicted_emoji_code)
  print(sent +  predicted_emoji)

### Stacked LSTM based model

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [ ]:
model = Sequential()

In [ ]:
model.add(LSTM(64, input_shape=(10, 50), return_sequences=True))
# model.add(Dropout(0.5))
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(64))
# model.add(Dropout(0.5))
model.add(Dense(5, activation='softmax')) 

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import ModelCheckpoint
model_checkpoint_callback = ModelCheckpoint('best_emoji_weights.h5', monitor='val_accuracy', save_best_only=True)

In [ ]:
hist = model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[model_checkpoint_callback])

In [ ]:
model.load_weights("best_emoji_weights.h5")

In [ ]:
plt.plot(hist.history["accuracy"])
plt.plot(hist.history["val_accuracy"])
plt.show()

In [ ]:
model.evaluate(X_train, y_train)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
pred = model.predict(X_test)
pred

In [ ]:
y_pred = np.argmax(pred, axis=1)
y_pred

In [ ]:
# print the prediction vs actual emoji 
for i, sent in enumerate(testing_corpus):
  print(sent + emoji.emojize(emoji_dictionary[str(y_pred[i])]) + emoji.emojize(emoji_dictionary[str(np.argmax(y_test, axis=1)[i])]))
  print()

In [ ]:
for i, sent in enumerate(testing_corpus):
  predicted_emoji_code = emoji_dictionary[str(y_pred[i])]
  predicted_emoji = emoji.emojize(predicted_emoji_code)
  print(sent +  predicted_emoji)